# Consistency across three rounds

The same 13 cases were run on all 24 variants three separate times, to measure whether the outcome of a query stays the same when it is asked again. All three rounds use the same agent model (`openai/gpt-oss-120b`, temperature 0) and the same judge (`openai/gpt-oss-120b`).

> The `judge_model` field in each `results.jsonl` row shows `ollama/ornith:9b` for run 1. That field is written by `deepeval/run.py` at collection time from its own default and is not the judge used for scoring. All rounds were scored by `deepeval/evaluate.py` with `openai/gpt-oss-120b`.

## 1. Files and settings

| Round | Raw input | Clean input used here |
|---|---|---|
| Run 1 | `../data/raw/baseline/run1/results.jsonl`, `scored.csv` | `../data/prepared/baseline_run1.csv` |
| Run 2 | `../data/raw/baseline/run2/results.jsonl`, `scored.csv` | `../data/prepared/baseline_run2.csv` (built in step 1b) |
| Run 3 | `../data/raw/baseline/run3/results.jsonl`, `scored.csv` | `../data/prepared/baseline_run3.csv` (built in step 1b) |

The comparison includes **all 13 cases: EVAL-01 to EVAL-13**, across **24 variants** and **3 rounds**. This gives **312 records per round** and **936 records in total**.

| Cases | Case type |
|---|---|
| EVAL-01 to EVAL-08 | Simple lookup |
| EVAL-09, EVAL-10, EVAL-12, EVAL-13 | Multi-step |
| EVAL-11 | Safety and guardrail |


In [1]:
from pathlib import Path
import io
import json
import math
import pandas as pd
import yaml

CASE_IDS = [f"EVAL-{i:02d}" for i in range(1, 14)]
KEY = ["case_id", "setup", "ticket_id"]
RUN_IDS = ["run_1", "run_2", "run_3"]
MARGINS = {"cost_usd": 0.10, "elapsed_s": 0.20}

cwd = Path.cwd()
candidates = [
    cwd,
    cwd / "results/deepeval/analysis",
    cwd / "GenAIProject/results/deepeval/analysis",
]
DIRECTORY = next(
    (p for p in candidates if (p / "1_consistency.ipynb").exists()),
    None,
)
if DIRECTORY is None:
    raise FileNotFoundError("Open the notebook from the project or its own directory.")

DEEPEVAL = DIRECTORY.parent
PREPARED = DEEPEVAL / "data" / "prepared"   # analysis inputs, one file per run
RAW = DEEPEVAL / "data" / "raw"             # untouched run output
TABLES = DEEPEVAL / "output" / "tables"     # results written by this notebook
CASES = DEEPEVAL.parents[1] / "deepeval" / "cases"

FILES = {
    "run_1": PREPARED / "baseline_run1.csv",
    "run_2": PREPARED / "baseline_run2.csv",
    "run_3": PREPARED / "baseline_run3.csv",
}
RERUN_FOLDERS = {
    "run_2": RAW / "baseline" / "run2",
    "run_3": RAW / "baseline" / "run3",
}

# Cost per run = tokens x the model price in configs/base.yaml (USD per million tokens).
llm = yaml.safe_load((DIRECTORY.parents[2] / "configs" / "base.yaml").read_text())["llm"]
models = set(llm["models"].values())
if len(models) != 1:
    raise ValueError(f"expected one model for every role, found {models}")
PRICE = llm["pricing"][models.pop()]

## 1b. Build run 2 and run 3 from the reruns

`data/raw/baseline/run2/` and `run3/` hold the raw DeepEval output (`scored.csv`) and the run logs (`results.jsonl`). This step turns each into a clean file with **exactly the same columns as run 1**, so all three rounds are processed identically.

| Column | How it is filled |
|---|---|
| `*_score`, `*_success`, `*_reason`, ticket statuses | Copied from `scored.csv` |
| `category` | Same labels as run 1 |
| `input_tokens`, `output_tokens` | Sum of every `llm_call` event in the run's trace (as for run 1) |
| `elapsed_s` | Sum of every `workflow_end` event (one per turn in multi-turn cases, as for run 1) |
| `*_success_manual` | `score >= 0.5`, the same rule used for run 1 |

**EVAL-11 and EVAL-13.** The ground truths of these two cases had been edited by mistake in `deepeval/cases/baseline.csv` before the reruns. They were restored to the run-1 wording in the rerun `results.jsonl` files, and the EVAL-11 and EVAL-13 rows were re-scored with the same judge (commit `e30b023`). The agents were not re-run: replies, tool calls, statuses, tokens and latency are unchanged. All three rounds are therefore scored against the same references, with no manual overrides.

The raw files in `data/raw/baseline/run2/` and `run3/` are never changed.


In [2]:
def read_trace_totals(path):
    # Same aggregation as run 1: tokens summed over every llm_call event, latency summed over every
    # workflow_end event (multi-turn cases EVAL-12 and EVAL-13 have one workflow_end per turn).
    rows = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            record = json.loads(line)
            events = record["trace"]["events"]
            calls = [e for e in events if e["kind"] == "llm_call"]
            ends = [e for e in events if e["kind"] == "workflow_end"]
            rows.append({
                "case_id": record["case_id"], "setup": record["setup"], "ticket_id": record["ticket_id"],
                "input_tokens": sum(e["input_tokens"] for e in calls),
                "output_tokens": sum(e["output_tokens"] for e in calls),
                "elapsed_s": round(sum(e["elapsed_s"] for e in ends), 4),
            })
    totals = pd.DataFrame(rows)
    if totals.duplicated(KEY).any():
        raise ValueError(f"{path}: duplicate case/setup/ticket keys")
    return totals


def read_run_1(path):
    # Older copies of run 1 were Excel workbooks saved with a .csv name; read either format.
    content = path.read_bytes()
    return pd.read_excel(io.BytesIO(content)) if content[:2] == b"PK" else pd.read_csv(path)


def build_clean_rerun(folder, run_id):
    run_1 = read_run_1(FILES["run_1"])
    scored = pd.read_csv(folder / "scored.csv")
    scored = scored.loc[scored.case_id.isin(CASE_IDS)]
    if scored.duplicated(KEY).any():
        raise ValueError(f"{folder}: duplicate case/setup/ticket keys")

    clean = scored.merge(read_trace_totals(folder / "results.jsonl"), on=KEY, how="left", validate="one_to_one")
    categories = run_1[["case_id", "category"]].drop_duplicates()   # same labels as run 1
    clean = clean.merge(categories, on="case_id", how="left", validate="many_to_one")
    if clean[["input_tokens", "category"]].isna().any().any():
        raise ValueError(f"{folder}: a scored row has no trace or category")

    # Manual labels: the same rule as run 1, score >= 0.5 (DeepEval's default threshold).
    for metric in MANUAL_METRICS:
        clean[f"{metric}_success_manual"] = clean[f"{metric}_score"] >= THRESHOLD

    columns = list(run_1.columns) + ["run_id"]
    clean["run_id"] = run_id
    clean = clean[columns].sort_values(KEY).reset_index(drop=True)
    if set(map(tuple, clean[KEY].to_numpy())) != set(map(tuple, run_1[KEY].to_numpy())):
        raise ValueError(f"{folder}: records differ from run 1")
    return clean


MANUAL_METRICS = ["answer_correctness", "tool_correctness", "argument_correctness", "step_efficiency_trace"]
THRESHOLD = 0.5

for run_id, folder in RERUN_FOLDERS.items():
    clean = build_clean_rerun(folder, run_id)
    clean.to_csv(FILES[run_id], index=False)
    print(f"{run_id}: {len(clean)} rows from {folder.relative_to(DEEPEVAL)} -> {FILES[run_id].name}")


run_2: 312 rows from data\raw\baseline\run2 -> baseline_run2.csv
run_3: 312 rows from data\raw\baseline\run3 -> baseline_run3.csv


## 2. Read and match the rounds

Match records using **case ID + setup + ticket ID**. Each round must have unique keys and the same selected records.

Only the four `*_success_manual` judgment columns are used. Task outcomes also include ticket-status agreement; tool outcomes combine tool and argument judgments. The non-manual success columns are ignored.


In [3]:
def read_input(path):
    # The baseline has a .csv name but currently contains an Excel workbook.
    content = path.read_bytes()
    if content[:2] == b"PK":
        return pd.read_excel(io.BytesIO(content))
    return pd.read_csv(path)


def as_boolean(values):
    lookup = {"true": True, "false": False, "1": True, "0": False}
    result = values.astype(str).str.strip().str.lower().map(lookup)
    if result.isna().any():
        raise ValueError(f"Missing or invalid manual judgment: {values.name}")
    return result.astype(bool)


def load_rounds():
    frames = []
    coverage = []
    baseline_keys = None
    manual_columns = [
        "answer_correctness_success_manual",
        "tool_correctness_success_manual",
        "argument_correctness_success_manual",
        "step_efficiency_trace_success_manual",
    ]

    for run_id, path in FILES.items():
        if not path.exists():
            coverage.append({"run_id": run_id, "status": "missing", "rows": 0})
            continue

        frame = read_input(path)
        frame = frame.loc[frame.case_id.isin(CASE_IDS)].copy()
        if frame[KEY].isna().any().any() or frame.duplicated(KEY).any():
            raise ValueError(f"{run_id}: missing or duplicate keys")

        keys = set(map(tuple, frame[KEY].to_numpy()))
        if baseline_keys is None:
            baseline_keys = keys
        elif keys != baseline_keys:
            raise ValueError(f"{run_id}: selected records differ from the other rounds")

        missing = set(manual_columns) - set(frame.columns)
        if missing:
            raise ValueError(f"{run_id}: missing manual judgments: {sorted(missing)}")

        frame["run_id"] = run_id
        frame["task_pass"] = (
            as_boolean(frame.answer_correctness_success_manual)
            & frame.ticket_status.eq(frame.expected_ticket_status)
        )
        frame["tool_pass"] = (
            as_boolean(frame.tool_correctness_success_manual)
            & as_boolean(frame.argument_correctness_success_manual)
        )
        frame["step_pass"] = as_boolean(frame.step_efficiency_trace_success_manual)

        for column in ["input_tokens", "output_tokens", "elapsed_s"]:
            values = pd.to_numeric(frame[column], errors="raise")
            if values.isna().any() or not values.map(math.isfinite).all() or values.lt(0).any():
                raise ValueError(f"{run_id}: invalid {column}")
            frame[column] = values
        frame["cost_usd"] = (
            frame.input_tokens * PRICE["input"] + frame.output_tokens * PRICE["output"]
        ) / 1_000_000

        frames.append(frame)
        coverage.append({"run_id": run_id, "status": "available", "rows": len(frame)})

    if not frames:
        raise ValueError("No rounds found")
    return pd.concat(frames, ignore_index=True), pd.DataFrame(coverage)


run_data, run_coverage = load_rounds()
print(run_coverage.to_string(index=False))

run_id    status  rows
 run_1 available   312
 run_2 available   312
 run_3 available   312


## 3. Boolean consistency: do all three outcomes match?

For a case, let b₁, b₂ and b₃ be its True/False outcomes.

$$
C_i = \begin{cases}
1 & \text{if } b_1=b_2=b_3 \\
0 & \text{otherwise}
\end{cases}
$$

| Run 1 | Run 2 | Run 3 | Consistent? |
|---|---|---|---|
| True | True | True | Yes |
| False | False | False | Yes |
| True | False | True | No |

Consistency measures agreement, not success. Apply this same rule separately to task, tool-call and step-efficiency outcomes.


In [4]:
def same_outcome(values):
    return bool(values.nunique() == 1)

## 4. Cost and latency stability: how far does a round move from the mean?

For one case, let x₁, x₂ and x₃ be its costs or times.

**Step A — calculate the three-round mean:**

$$
\bar{x}=\frac{x_1+x_2+x_3}{3}
$$

**Step B — calculate the largest percentage distance from that mean:**

$$
D_i=\frac{\max\left(|x_1-\bar{x}|,|x_2-\bar{x}|,|x_3-\bar{x}|\right)}{\bar{x}}\times100\%
$$

**Step C — compare with the chosen margin:**

| Measure | Stable if | Meaning |
|---|---|---|
| Cost (USD) | Dᵢ ≤ 10% | Every round is within ±10% of its mean |
| Elapsed time | Dᵢ ≤ 20% | Every round is within ±20% of its mean |

Apply the formula separately to cost and elapsed time. Cost per run is input tokens × input price + output tokens × output price, using the model price in `configs/base.yaml` (`llm.pricing`, USD per million tokens). If all three values are zero, deviation is 0%.


In [5]:
def maximum_deviation(values):
    mean = values.mean()
    if mean == 0:
        return 0.0
    return float((values - mean).abs().max() / mean)

## 5. Calculate one result per case and setup

For each metric, compare the three rounds within the same case. Missing rounds leave consistency and stability results blank.


In [6]:
def build_case_results(data):
    records = []
    for key, group in data.groupby(KEY, sort=True):
        complete = set(group.run_id) == set(RUN_IDS)
        result = dict(zip(KEY, key))
        result.update(runs_available=len(group), complete=complete)

        for metric in ["task", "tool", "step"]:
            result[f"{metric}_same_outcome"] = (
                same_outcome(group[f"{metric}_pass"]) if complete else None
            )

        for metric, margin in MARGINS.items():
            values = group[metric]
            deviation = maximum_deviation(values)
            result[f"{metric}_mean"] = values.mean()
            result[f"{metric}_max_abs_deviation_pct"] = deviation * 100 if complete else None
            result[f"{metric}_within_margin"] = (
                bool(deviation <= margin + 1e-12) if complete else None
            )
        records.append(result)

    return pd.DataFrame(records)


case_results = build_case_results(run_data)
print(f"Calculated {len(case_results)} case/setup results.")

Calculated 312 case/setup results.


## 6. Combine cases into one summary row per setup

Let **N = 13** be the number of selected cases for each setup.

**Boolean consistency:**

$$
\text{Consistency (\%)}=\frac{\text{Number of cases with identical outcomes in all 3 rounds}}{N}\times100\%
$$

**Cost or latency stability:**

$$
\text{Stable cases (\%)}=\frac{\text{Number of cases within the chosen deviation margin}}{N}\times100\%
$$

For example, **12 consistent cases out of 13 = 92.31% consistency**. If **11 cases stay within the latency margin**, latency stability is **84.62%**.

| Summary column | What is counted? |
|---|---|
| `task_success_consistency_pct` | Cases with the same task outcome across all three rounds |
| `tool_call_accuracy_consistency_pct` | Cases with the same tool-call outcome across all three rounds |
| `step_efficiency_consistency_pct` | Cases with the same efficiency outcome across all three rounds |
| `cost_usd_stable_cases_pct` | Cases within the 10% cost margin |
| `elapsed_s_stable_cases_pct` | Cases within the 20% latency margin |

Calculate the deviation **within each case first**. Do not compare costs or times between different cases.


In [7]:
def build_setup_summary(details):
    metric_labels = {
        "task": "task_success",
        "tool": "tool_call_accuracy",
        "step": "step_efficiency",
    }
    records = []
    for setup, group in details.groupby("setup", sort=True):
        complete = bool(group.complete.all()) and set(group.case_id) == set(CASE_IDS)
        result = {
            "setup": setup,
            "cases_available": len(group),
            "cases_expected": len(CASE_IDS),
            "complete_cases": int(group.complete.sum()),
            "status": "complete" if complete else "awaiting runs or cases",
        }

        for metric, label in metric_labels.items():
            result[f"{label}_consistency_pct"] = (
                group[f"{metric}_same_outcome"].mean() * 100 if complete else None
            )
        for metric in MARGINS:
            result[f"{metric}_stable_cases_pct"] = (
                group[f"{metric}_within_margin"].mean() * 100 if complete else None
            )
        records.append(result)
    return pd.DataFrame(records)


setup_summary = build_setup_summary(case_results)
print(setup_summary.drop(columns=["cases_expected", "complete_cases", "status"]).round(2).to_string(index=False))

setup  cases_available  task_success_consistency_pct  tool_call_accuracy_consistency_pct  step_efficiency_consistency_pct  cost_usd_stable_cases_pct  elapsed_s_stable_cases_pct
 s1-a               13                         69.23                               92.31                            84.62                      92.31                        0.00
 s1-b               13                         84.62                               84.62                            76.92                      23.08                       23.08
 s2-a               13                         92.31                               92.31                            92.31                      69.23                        7.69
 s2-b               13                         69.23                               92.31                            76.92                      30.77                       30.77
 s3-a               13                         61.54                               92.31                           

## 7. Headline metrics in each round

Consistency (step 6) asks whether each *case* repeats its outcome. This table asks the coarser question: do the headline numbers of the evaluation hold when everything is run again? Each row is one full round of 312 records, scored with the same definitions as the final evaluation.

| Column | Definition |
|---|---|
| `task_success_pct` | % of records where the answer passes (manual label) **and** the ticket status matches |
| `tool_call_accuracy_pct` | % of records where tool selection **and** arguments pass |
| `step_efficiency_pct` | % of records where step efficiency passes |
| `*_mean_score` | Mean raw judge score (0-1) for tool correctness, argument correctness and step efficiency |
| `median_cost_usd`, `median_latency_s` | Medians per record |


In [8]:
by_run = run_data.groupby("run_id").agg(
    records=("case_id", "size"),
    task_success_pct=("task_pass", "mean"),
    tool_call_accuracy_pct=("tool_pass", "mean"),
    step_efficiency_pct=("step_pass", "mean"),
    tool_correctness_mean_score=("tool_correctness_score", "mean"),
    argument_correctness_mean_score=("argument_correctness_score", "mean"),
    step_efficiency_mean_score=("step_efficiency_trace_score", "mean"),
    median_cost_usd=("cost_usd", "median"),
    median_latency_s=("elapsed_s", "median"),
)
for column in ["task_success_pct", "tool_call_accuracy_pct", "step_efficiency_pct"]:
    by_run[column] *= 100
by_run.loc["mean of 3 runs"] = by_run.mean()
by_run.loc["range (max - min)"] = by_run.iloc[:3].max() - by_run.iloc[:3].min()
by_run.round({"task_success_pct": 1, "tool_call_accuracy_pct": 1, "step_efficiency_pct": 1,
              "tool_correctness_mean_score": 3, "argument_correctness_mean_score": 3,
              "step_efficiency_mean_score": 3, "median_cost_usd": 6, "median_latency_s": 1})


,records,task_success_pct,tool_call_accuracy_pct,step_efficiency_pct,tool_correctness_mean_score,argument_correctness_mean_score,step_efficiency_mean_score,median_cost_usd,median_latency_s
run_id,,,,,,,,,
run_1,312.0,86.2,83.3,95.8,0.912,0.826,0.911,0.000494,33.8
run_2,312.0,89.4,85.3,91.0,0.924,0.833,0.877,0.000530,22.7
run_3,312.0,88.5,86.2,89.1,0.926,0.819,0.888,0.000502,29.5
mean of 3 runs,312.0,88.0,84.9,92.0,0.921,0.826,0.892,0.000509,28.7
range (max - min),0.0,3.2,2.9,6.7,0.014,0.014,0.034,0.000037,11.1


## 8. Pairwise agreement

The evaluation plan defines consistency as `original_pass == second_pass`. This table gives that agreement for each pair of rounds; step 6 uses the stricter rule that all three rounds agree.


In [9]:
wide = run_data.pivot_table(index=KEY, columns="run_id", values=["task_pass", "tool_pass", "step_pass"], aggfunc="first")
pairwise = pd.DataFrame({
    f"{a} vs {b}": {metric: (wide[(f"{metric}_pass", a)] == wide[(f"{metric}_pass", b)]).mean() * 100
                    for metric in ["task", "tool", "step"]}
    for a, b in [("run_1", "run_2"), ("run_1", "run_3"), ("run_2", "run_3")]
})
pairwise["all three agree"] = [case_results[f"{m}_same_outcome"].mean() * 100 for m in ["task", "tool", "step"]]
pairwise.index = ["task success", "tool-call accuracy", "step efficiency"]
pairwise.round(1)


,run_1 vs run_2,run_1 vs run_3,run_2 vs run_3,all three agree
task success,85.9,85.6,89.4,80.4
tool-call accuracy,90.4,90.7,90.7,85.9
step efficiency,90.1,88.8,87.2,83.0


## 9. Consistency by design dimension and case category

Pooled over the 12 variants on each side of a design choice (156 records per side; for agent count 104 single vs 208 multi). The agentic-vs-graph row is also tested on matched pairs (same agent count, policy, variant and case, e.g. s1-a vs s5-a on EVAL-09) with an exact McNemar test, because hypothesis H2 predicts that graph control gives more consistent results.


In [10]:
from scipy import stats


def describe_setup(setup_id):
    family, variant = int(setup_id[1]), setup_id[-1]
    return {
        "setup": setup_id,
        "control": "agentic" if family <= 4 else "graph",
        "agents": "single" if family in (1, 2, 5, 6) else "multi",
        "policy": "rag" if family % 2 == 1 else "skills",
        "strategy (a vs b)": {"a": "react", "b": "plan_execute"}.get(variant),
        "pair_id": f"{'single' if family in (1, 2, 5, 6) else 'multi'}-{family % 2}-{variant}",
    }


labelled = case_results.merge(pd.DataFrame([describe_setup(s) for s in case_results.setup.unique()]), on="setup")
labelled = labelled.merge(pd.read_csv(CASES / "categories.csv")[["case_id", "category"]], on="case_id")
SAME = {"task_same_outcome": "task_consistency_pct", "tool_same_outcome": "tool_consistency_pct",
        "step_same_outcome": "step_consistency_pct", "cost_usd_within_margin": "cost_stable_pct",
        "elapsed_s_within_margin": "latency_stable_pct"}

frames = []
for dimension in ["control", "agents", "policy", "strategy (a vs b)", "category"]:
    table = labelled.dropna(subset=[dimension]).groupby(dimension)[list(SAME)].mean().mul(100).rename(columns=SAME)
    table.insert(0, "records", labelled.dropna(subset=[dimension]).groupby(dimension).size())
    frames.append(table.reset_index().rename(columns={dimension: "group"}).assign(dimension=dimension))
by_group = pd.concat(frames, ignore_index=True)[["dimension", "group", "records", *SAME.values()]]
display(by_group.round(1))

# Matched agentic/graph pairs: does graph control repeat its outcome more often?
tests = []
for metric in ["task", "tool", "step"]:
    paired = labelled.pivot_table(index=["pair_id", "case_id"], columns="control",
                                  values=f"{metric}_same_outcome", aggfunc="first").astype(float).dropna()
    only_graph = int(((paired.graph == 1) & (paired.agentic == 0)).sum())
    only_agentic = int(((paired.graph == 0) & (paired.agentic == 1)).sum())
    p = stats.binomtest(only_graph, only_graph + only_agentic, 0.5).pvalue if only_graph + only_agentic else 1.0
    tests.append({"metric": metric, "pairs": len(paired), "graph consistent %": paired.graph.mean() * 100,
                  "agentic consistent %": paired.agentic.mean() * 100,
                  "only graph consistent": only_graph, "only agentic consistent": only_agentic, "McNemar p": p})
consistency_tests = pd.DataFrame(tests).set_index("metric")
consistency_tests.round(3)


,dimension,group,records,task_consistency_pct,tool_consistency_pct,step_consistency_pct,cost_stable_pct,latency_stable_pct
0,control,agentic,156,75.6,89.1,83.3,43.6,20.5
1,control,graph,156,85.3,82.7,82.7,50.6,26.3
2,agents,multi,208,80.3,84.6,83.2,45.2,24.0
3,agents,single,104,80.8,88.5,82.7,51.0,22.1
4,policy,rag,156,77.6,86.5,83.3,48.1,22.4
5,policy,skills,156,83.3,85.3,82.7,46.2,24.4
6,strategy (a vs b),plan_execute,104,80.8,85.6,82.7,35.6,32.7
7,strategy (a vs b),react,104,76.9,82.7,81.7,54.8,17.3
8,category,multi-step,96,63.5,59.4,60.4,17.7,29.2
9,category,safety & guardrail,24,100.0,100.0,83.3,45.8,4.2


,pairs,graph consistent %,agentic consistent %,only graph consistent,only agentic consistent,McNemar p
metric,,,,,,
task,156,85.256,75.641,27,12,0.024
tool,156,82.692,89.103,3,13,0.021
step,156,82.692,83.333,19,20,1.000


## 10. Which cases changed outcome?

Number of variants (out of 24) whose outcome was **not** the same in all three rounds, per case. This shows where the inconsistency comes from.


In [11]:
by_case = labelled.groupby("case_id").agg(
    category=("category", "first"),
    task_changed=("task_same_outcome", lambda s: int((~s.astype(bool)).sum())),
    tool_changed=("tool_same_outcome", lambda s: int((~s.astype(bool)).sum())),
    step_changed=("step_same_outcome", lambda s: int((~s.astype(bool)).sum())),
    latency_unstable=("elapsed_s_within_margin", lambda s: int((~s.astype(bool)).sum())),
)
display(by_case)


,category,task_changed,tool_changed,step_changed,latency_unstable
case_id,,,,,
EVAL-01,simple lookup,3,0,1,16
EVAL-02,simple lookup,3,0,0,19
EVAL-03,simple lookup,6,0,1,22
EVAL-04,simple lookup,9,3,5,16
EVAL-05,simple lookup,5,2,3,19
EVAL-06,simple lookup,0,0,1,19
EVAL-07,simple lookup,0,0,0,18
EVAL-08,simple lookup,0,0,0,19
EVAL-09,multi-step,12,6,11,15


## 11. Save the results

| Output | Contents |
|---|---|
| `data/prepared/baseline_run2.csv`, `baseline_run3.csv` | Prepared inputs for runs 2 and 3 (step 1b) |
| `consistency_by_setup_and_case.csv` | One comparison row per case and variant across all three rounds; not a raw concatenation |
| `consistency_by_setup.csv` | Consistency and stability percentages for each setup (read by `3_scorecard`) |
| `consistency_pass_rates_by_run.csv` | Headline metrics in each round (step 7) |
| `consistency_by_architecture.csv` | Consistency by design dimension and case category (step 9) |
| `consistency_changes_by_case.csv` | Variants whose outcome changed, per case (step 10) |

The raw files in `data/raw/baseline/run2/` and `run3/` are unchanged. Rerun all cells after changing files, cases or margins, then rerun `3_scorecard.ipynb`.


In [12]:
case_results.to_csv(TABLES / "consistency_by_setup_and_case.csv", index=False)
setup_summary.to_csv(TABLES / "consistency_by_setup.csv", index=False)
by_run.to_csv(TABLES / "consistency_pass_rates_by_run.csv")
by_group.to_csv(TABLES / "consistency_by_architecture.csv", index=False)
by_case.to_csv(TABLES / "consistency_changes_by_case.csv")
print("Saved the result tables.")


Saved the result tables.
